In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("HomeCredit_Serving_Layer") \
    .enableHiveSupport() \
    .getOrCreate()

In [2]:
raw_app = spark.read.parquet("/user/student/home_credit/raw/application_train")
raw_prev = spark.read.parquet("/user/student/home_credit/raw/previous_application")
raw_inst = spark.read.parquet("/user/student/home_credit/raw/installments_payments")
raw_bureau = spark.read.parquet("/user/student/home_credit/raw/bureau")

print("Raw Parquet Data loaded successfully from HDFS!")

Raw Parquet Data loaded successfully from HDFS!


In [3]:
# 1. Application Train
# Current loan application + applicant profile
df_app = raw_app.select(
    F.col("SK_ID_CURR").cast("int"),                 # Unique ID of the current loan application/customer
    F.col("TARGET").cast("int"),                     # Loan outcome: 1 = payment difficulty/default risk, 0 = no difficulty
    F.col("NAME_CONTRACT_TYPE").cast("string"),      # Type of current loan/credit contract
    F.col("DAYS_BIRTH").cast("int"),                 # Applicant age in days before the application date
    F.col("OCCUPATION_TYPE").cast("string"),         # Applicant's occupation/job type
    F.col("NAME_EDUCATION_TYPE").cast("string"),     # Applicant's education level
    F.col("NAME_FAMILY_STATUS").cast("string"),      # Marital/family status
    F.col("NAME_HOUSING_TYPE").cast("string"),       # Housing situation: own house, rent, parents, etc.
    F.col("NAME_INCOME_TYPE").cast("string"),        # Main source/type of income
    F.col("FLAG_OWN_REALTY").cast("string"),         # Whether applicant owns property: Y/N
    F.col("AMT_INCOME_TOTAL").cast("double"),        # Applicant's total reported income
    F.col("AMT_CREDIT").cast("double"),              # Amount of credit granted for the current loan
    F.col("AMT_ANNUITY").cast("double"),             # Scheduled periodic payment for the current loan
    F.col("AMT_GOODS_PRICE").cast("double"),         # Price of goods being financed
    F.col("DAYS_EMPLOYED").cast("int")               # Employment duration in days before application
)


# 2. Previous Applications
# Previous Home Credit loan applications made by the same customer
df_prev = raw_prev.select(
    F.col("SK_ID_PREV").cast("int"),                 # Unique ID of the previous loan application
    F.col("SK_ID_CURR").cast("int"),                 # Links previous application to the current customer
    F.col("NAME_CONTRACT_TYPE").cast("string"),      # Type of previous loan/credit contract
    F.col("NAME_CONTRACT_STATUS").cast("string"),    # Previous application status: Approved, Refused, etc.
    F.col("AMT_APPLICATION").cast("double"),         # Amount originally requested by the customer
    F.col("AMT_CREDIT").cast("double"),              # Amount actually financed/granted
    F.col("AMT_ANNUITY").cast("double"),             # Scheduled payment amount for the previous loan
    F.col("DAYS_DECISION").cast("int"),              # Days between previous decision and current application
    F.col("CNT_PAYMENT").cast("double")              # Planned number of payments/installments
)


# 3. Installments Payments
# Actual payment behavior for previous Home Credit loans
df_inst = raw_inst.select(
    F.col("SK_ID_PREV").cast("int"),                  # Previous Home Credit loan
    F.col("SK_ID_CURR").cast("int"),                  # Current customer/application
    F.col("NUM_INSTALMENT_VERSION").cast("double"),   # Version of installment schedule
    F.col("NUM_INSTALMENT_NUMBER").cast("int"),       # Installment sequence number
    F.col("DAYS_INSTALMENT").cast("double"),          # Scheduled payment date
    F.col("DAYS_ENTRY_PAYMENT").cast("double"),       # Actual payment date
    F.col("AMT_INSTALMENT").cast("double"),           # Expected installment amount
    F.col("AMT_PAYMENT").cast("double")               # Actual payment amount
)

# 4. Bureau Data
# Customer credit history reported by other lenders through the credit bureau
df_bureau = raw_bureau.select(
    F.col("SK_ID_BUREAU").cast("int"),               # Unique ID of an external bureau credit record
    F.col("SK_ID_CURR").cast("int"),                 # Links bureau credit to the current customer
    F.col("CREDIT_ACTIVE").cast("string"),           # Status of external credit: Active, Closed, etc.
    F.col("CREDIT_DAY_OVERDUE").cast("int"),          # Number of days the external credit is overdue
    F.col("AMT_CREDIT_SUM").cast("double"),           # Total amount of the external credit
    F.col("AMT_CREDIT_SUM_DEBT").cast("double"),      # Remaining outstanding debt
    F.col("AMT_CREDIT_SUM_OVERDUE").cast("double"),   # Amount currently overdue
)

print("Data selected successfully!")

Data selected successfully!


| Table                   | Grain before processing                                | Simple meaning                                                                         |
| ----------------------- | ------------------------------------------------------ | -------------------------------------------------------------------------------------- |
| `application_train`     | **1 row per current application (`SK_ID_CURR`)**       | Each row describes one customer's current Home Credit application.                     |
| `previous_application`  | **1 row per previous application (`SK_ID_PREV`)**      | One customer (`SK_ID_CURR`) can have many previous Home Credit applications.           |
| `installments_payments` | **1 row per payment transaction for an installment**   | One installment can have multiple payment rows if it was paid in several transactions. |
| `bureau`                | **1 row per external credit account (`SK_ID_BUREAU`)** | One customer can have many loans/credit accounts reported by other lenders.            |


## Profile the selected data

In [17]:
df_app.printSchema()
print(df_app.count())
print(df_app.select("SK_ID_CURR").distinct().count())


root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- OCCUPATION_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)

307511


307511


In [19]:
df_prev.printSchema()
print(df_prev.count())
print(df_prev.select("SK_ID_CURR").distinct().count())


root
 |-- SK_ID_PREV: integer (nullable = true)
 |-- SK_ID_CURR: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- NAME_CONTRACT_STATUS: string (nullable = true)
 |-- AMT_APPLICATION: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- DAYS_DECISION: integer (nullable = true)
 |-- CNT_PAYMENT: double (nullable = true)

1670214
338857



[Stage 52:==========================================>           (159 + 6) / 200]



In [52]:
df_inst.printSchema()
print(df_inst.count())
print(df_inst.select("SK_ID_CURR").distinct().count())


root
 |-- SK_ID_PREV: integer (nullable = true)
 |-- SK_ID_CURR: integer (nullable = true)
 |-- NUM_INSTALMENT_NUMBER: integer (nullable = true)
 |-- DAYS_INSTALMENT: double (nullable = true)
 |-- DAYS_ENTRY_PAYMENT: double (nullable = true)
 |-- AMT_INSTALMENT: double (nullable = true)
 |-- AMT_PAYMENT: double (nullable = true)

13605401


339587



[Stage 121:===============================================>         (5 + 1) / 6]

[Stage 122:==================================================>  (189 + 8) / 200]



In [ ]:
df_inst.where(F.col("SK_ID_PREV")==1054186).orderBy("NUM_INSTALMENT_NUMBER").show()

+----------+----------+---------------------+---------------+------------------+--------------+-----------+
|SK_ID_PREV|SK_ID_CURR|NUM_INSTALMENT_NUMBER|DAYS_INSTALMENT|DAYS_ENTRY_PAYMENT|AMT_INSTALMENT|AMT_PAYMENT|
+----------+----------+---------------------+---------------+------------------+--------------+-----------+
|   1054186|    161674|                    1|        -1330.0|           -1338.0|       6948.36|    6948.36|
|   1054186|    161674|                    2|        -1300.0|           -1307.0|       6948.36|    6948.36|
|   1054186|    161674|                    3|        -1270.0|           -1275.0|       6948.36|    6948.36|
|   1054186|    161674|                    4|        -1240.0|           -1247.0|       6948.36|    6948.36|
|   1054186|    161674|                    5|        -1210.0|           -1217.0|       6948.36|    6948.36|
|   1054186|    161674|                    6|        -1180.0|           -1187.0|       6948.36|    6948.36|
|   1054186|    161674|     

In [23]:
df_bureau.printSchema()
print(df_bureau.count())
print(df_bureau.select("SK_ID_CURR").distinct().count())

root
 |-- SK_ID_BUREAU: integer (nullable = true)
 |-- SK_ID_CURR: integer (nullable = true)
 |-- CREDIT_ACTIVE: string (nullable = true)
 |-- CREDIT_TYPE: string (nullable = true)
 |-- DAYS_CREDIT: integer (nullable = true)
 |-- CREDIT_DAY_OVERDUE: integer (nullable = true)
 |-- AMT_CREDIT_SUM: double (nullable = true)
 |-- AMT_CREDIT_SUM_DEBT: double (nullable = true)
 |-- AMT_CREDIT_SUM_OVERDUE: double (nullable = true)
 |-- AMT_CREDIT_MAX_OVERDUE: double (nullable = true)

1716428
305811



[Stage 72:================================================>     (178 + 9) / 200]



## Create useful derived features From `application_train`

In [32]:
df_app.filter(F.col("DAYS_EMPLOYED") == 365243).count()

55374

In [28]:
df_app_features = (
    df_app

    # Employment duration relative to applicant age
    # 365243 = 1000 year is a special/anomalous Home Credit value,
    # so it should not be interpreted as real employment duration.
    # treating the special value as NULL
    
    .withColumn(
        "Employed_to_Age",

        F.when(
            (F.col("DAYS_EMPLOYED") == 365243) |
            F.col("DAYS_EMPLOYED").isNull() |
            F.col("DAYS_BIRTH").isNull() |
            (F.col("DAYS_BIRTH") == 0),
            F.lit(None).cast("double")
        )

        .otherwise(
            F.round(
                F.abs(
                    F.col("DAYS_EMPLOYED") /
                    F.col("DAYS_BIRTH")
                ),
                2
            )
        )
    )
    # Loan amount relative to applicant income
    .withColumn(
        "Credit_to_Income",
        F.when(
            F.col("AMT_INCOME_TOTAL") > 0,
            F.round(
                F.col("AMT_CREDIT") / F.col("AMT_INCOME_TOTAL"),
                3
            )
        )
    )

    # Scheduled loan payment relative to applicant income
    .withColumn(
        "Annuity_to_Income",
        F.when(
            F.col("AMT_INCOME_TOTAL") > 0,
            F.round(
                F.col("AMT_ANNUITY") / F.col("AMT_INCOME_TOTAL"),
                3
            )
        )
    )

    # Credit amount relative to the price of financed goods
    .withColumn(
        "LTV",
        F.when(
            F.col("AMT_GOODS_PRICE") > 0,
            F.round(
                F.col("AMT_CREDIT") / F.col("AMT_GOODS_PRICE"),
                3
            )
        )
    )

    # Approximate relationship between credit amount and scheduled payment
    .withColumn(
        "Credit_to_Annuity",
        F.when(
            F.col("AMT_ANNUITY") > 0,
            F.round(
                F.col("AMT_CREDIT") / F.col("AMT_ANNUITY"),
                2
            )
        )
    )
)

In [29]:
df_app_features.select(
    "SK_ID_CURR",
    "Employed_to_Age",
    "Credit_to_Income",
    "Annuity_to_Income",
    "LTV",
    "Credit_to_Annuity"
).show(10)

+----------+---------------+----------------+-----------------+-----+-----------------+
|SK_ID_CURR|Employed_to_Age|Credit_to_Income|Annuity_to_Income|  LTV|Credit_to_Annuity|
+----------+---------------+----------------+-----------------+-----+-----------------+
|    100002|           0.07|           2.008|            0.122|1.158|            16.46|
|    100003|           0.07|           4.791|            0.132|1.145|            36.23|
|    100004|           0.01|             2.0|              0.1|  1.0|             20.0|
|    100006|           0.16|           2.316|             0.22|1.053|            10.53|
|    100007|           0.15|           4.222|             0.18|  1.0|            23.46|
|    100008|           0.09|           4.955|            0.278|1.079|            17.82|
|    100009|           0.23|           9.127|            0.242|1.119|            37.79|
|    100010|           0.02|            4.25|            0.117|  1.0|            36.36|
|    100011|           null|    

In [33]:
df_app_features.filter(
    F.col("Employed_to_Age") >1 
).count()

0

## Create useful derived features From `previous_application`

In [34]:
# ============================================================
# Check Contract Status Values
# ============================================================

df_prev.groupBy("NAME_CONTRACT_STATUS") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

+--------------------+-------+
|NAME_CONTRACT_STATUS|  count|
+--------------------+-------+
|            Approved|1036781|
|            Canceled| 316319|
|             Refused| 290678|
|        Unused offer|  26436|
+--------------------+-------+



In [38]:
# ============================================================
# Previous Applications - Customer Level Aggregation
#
# Goal:
# Convert many previous-application rows per customer
# into exactly ONE row per SK_ID_CURR.
# ============================================================

prev_agg = (
    df_prev

    .groupBy("SK_ID_CURR")

    .agg(

        # Total number of previous applications
        F.countDistinct("SK_ID_PREV")
         .alias("Prev_App_Count"),

        # Number of previous approved applications
        F.countDistinct(
            F.when(
                F.col("NAME_CONTRACT_STATUS") == "Approved",
                F.col("SK_ID_PREV")
            )
        ).alias("Approved_App_Count"),

        # Number of previous refused applications
        F.countDistinct(
            F.when(
                F.col("NAME_CONTRACT_STATUS") == "Refused",
                F.col("SK_ID_PREV")
            )
        ).alias("Refused_App_Count"),

        # Total credit amount across previous applications
        F.round(
            F.sum("AMT_CREDIT"),
            2
        ).alias("Prev_Total_Credit"),

        # Average previous credit amount
        F.round(
            F.avg("AMT_CREDIT"),
            2
        ).alias("Prev_Avg_Credit"),

        # Average scheduled payment amount
        F.round(
            F.avg("AMT_ANNUITY"),
            2
        ).alias("Prev_Avg_Annuity"),

        # Average planned number of installments
        F.round(
            F.avg("CNT_PAYMENT"),
            1
        ).alias("Prev_Avg_Payment_Term"),

        # Most recent previous application:
        # DAYS_DECISION is negative, so MAX is closest to zero.
        F.abs(
            F.max("DAYS_DECISION")
        ).alias("Days_Since_Last_Prev_App")
    )
)

In [39]:
# ============================================================
# Previous Applications - Derived Ratios
# ============================================================

prev_agg = (
    prev_agg

    # Ratio of previous applications that were approved
    .withColumn(
        "Approved_App_Ratio",
        F.when(
            F.col("Prev_App_Count") > 0,
            F.round(
                F.col("Approved_App_Count") /
                F.col("Prev_App_Count"),
                3
            )
        )
    )

    # Ratio of previous applications that were refused
    .withColumn(
        "Refused_App_Ratio",
        F.when(
            F.col("Prev_App_Count") > 0,
            F.round(
                F.col("Refused_App_Count") /
                F.col("Prev_App_Count"),
                3
            )
        )
    )
)

In [40]:
# ============================================================
# Previous Applications - Preview Aggregated Features
# ============================================================

prev_agg.select(
    "SK_ID_CURR",
    "Prev_App_Count",
    "Approved_App_Ratio",
    "Refused_App_Ratio",
    "Prev_Total_Credit",
    "Prev_Avg_Credit",
    "Prev_Avg_Annuity",
    "Prev_Avg_Payment_Term",
    "Days_Since_Last_Prev_App"
).show(10, truncate=False)

2026-09-04 21:10:21,784 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:10:21,922 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:10:24,199 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+----------+--------------+------------------+-----------------+-----------------+---------------+----------------+---------------------+------------------------+
|SK_ID_CURR|Prev_App_Count|Approved_App_Ratio|Refused_App_Ratio|Prev_Total_Credit|Prev_Avg_Credit|Prev_Avg_Annuity|Prev_Avg_Payment_Term|Days_Since_Last_Prev_App|
+----------+--------------+------------------+-----------------+-----------------+---------------+----------------+---------------------+------------------------+
|104688    |5             |1.0               |0.0              |607338.0         |121467.6       |11807.76        |14.4                 |808                     |
|109068    |2             |0.5               |0.0              |51147.0          |25573.5        |4684.05         |12.0                 |178                     |
|124967    |1             |1.0               |0.0              |31455.0          |31455.0        |3736.85         |12.0                 |2080                    |
|130544    |2         

In [41]:
# ============================================================
# Validate aggregation grain
# ============================================================

duplicate_customers = (
    prev_agg
    .groupBy("SK_ID_CURR")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate SK_ID_CURR values:", duplicate_customers)

Duplicate SK_ID_CURR values: 0


## Create useful derived features From `installments_payments`

In [42]:
# df_inst is not guaranteed to be one row per installment.
# Check whether one installment has multiple payment records


duplicate_installments = (
    df_inst
    .groupBy(
        "SK_ID_PREV",
        "NUM_INSTALMENT_VERSION",
        "NUM_INSTALMENT_NUMBER"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_installments.show(20, truncate=False)

2026-09-04 21:12:02,662 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:12:03,048 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:12:03,061 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:12:03,084 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:12:03,252 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:12:03,262 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:12:04,099 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:12:04,158 WARN expressions.RowBasedKeyVal

+----------+----------------------+---------------------+-----+
|SK_ID_PREV|NUM_INSTALMENT_VERSION|NUM_INSTALMENT_NUMBER|count|
+----------+----------------------+---------------------+-----+
|2574354   |1.0                   |6                    |2    |
|1102942   |1.0                   |1                    |2    |
|1213567   |1.0                   |13                   |2    |
|2317809   |1.0                   |23                   |2    |
|1036397   |1.0                   |7                    |2    |
|2053133   |1.0                   |6                    |2    |
|2527423   |1.0                   |12                   |2    |
|1492980   |1.0                   |6                    |2    |
|1107760   |1.0                   |3                    |2    |
|2643068   |1.0                   |16                   |2    |
|2300078   |1.0                   |14                   |2    |
|2410097   |1.0                   |3                    |2    |
|2013807   |2.0                   |35   

In [43]:
# Example of one installment has multiple payment records.

df_inst.filter(
    (F.col("SK_ID_PREV") == 2013807) 
).orderBy("NUM_INSTALMENT_NUMBER").show(truncate=False)

# For example, installment 5 appears twice: 
# They had one installment of 23,787.14 that was paid through two separate payment transactions.
# One row = one payment transaction against an installment, not necessarily one row per installment.

+----------+----------+----------------------+---------------------+---------------+------------------+--------------+-----------+
|SK_ID_PREV|SK_ID_CURR|NUM_INSTALMENT_VERSION|NUM_INSTALMENT_NUMBER|DAYS_INSTALMENT|DAYS_ENTRY_PAYMENT|AMT_INSTALMENT|AMT_PAYMENT|
+----------+----------+----------------------+---------------------+---------------+------------------+--------------+-----------+
|2013807   |174364    |1.0                   |1                    |-1124.0        |-1126.0           |23787.14      |23787.14   |
|2013807   |174364    |1.0                   |2                    |-1094.0        |-1095.0           |23787.14      |23787.14   |
|2013807   |174364    |1.0                   |3                    |-1064.0        |-1066.0           |23787.14      |23787.14   |
|2013807   |174364    |1.0                   |4                    |-1034.0        |-1066.0           |23787.14      |23787.14   |
|2013807   |174364    |1.0                   |5                    |-1004.0        

In [59]:
# ============================================================
# Aggregate payment transactions to installment level
#
# One installment can be paid using multiple transactions.
# This prevents duplicate counting of the expected amount.
# ============================================================

inst_level = (
    df_inst
    .groupBy(
        "SK_ID_CURR",
        "SK_ID_PREV",
        "NUM_INSTALMENT_VERSION",
        "NUM_INSTALMENT_NUMBER"
    )
    .agg(
        # Scheduled due date of the installment
        F.min("DAYS_INSTALMENT").alias("DAYS_INSTALMENT"),

        # Date when the installment was finally paid
        F.max("DAYS_ENTRY_PAYMENT").alias("DAYS_ENTRY_PAYMENT"),

        # Expected amount should only be counted once
        F.max("AMT_INSTALMENT").alias("AMT_INSTALMENT"),

        # Sum all partial payments for this installment
        F.sum("AMT_PAYMENT").alias("AMT_PAYMENT")
    )
)

In [60]:
# ============================================================
# Installments Payments - Installment-Level Features
#
# Grain:
# One row = one installment
#
# Features created:
#   1. Days_Past_Due
#   2. Is_Late
#   3. Underpaid_Amount
# ============================================================

inst_level = (
    inst_level

    # Number of days the installment was paid late.
    # If paid early or on time, store 0.
    .withColumn(
        "Days_Past_Due",
        F.greatest(
            F.col("DAYS_ENTRY_PAYMENT") - F.col("DAYS_INSTALMENT"),
            F.lit(0.0)
        )
    )

    # Flag late installments:
    # 1 = paid late
    # 0 = paid on time or early
    .withColumn(
        "Is_Late",
        F.when(
            F.col("Days_Past_Due") > 0,
            1
        ).otherwise(0)
    )

    # Amount still missing from the expected installment.
    # If fully paid or overpaid, store 0.
    .withColumn(
        "Underpaid_Amount",
        F.greatest(
            F.col("AMT_INSTALMENT") - F.col("AMT_PAYMENT"),
            F.lit(0.0)
        )
    )
)

In [61]:
# ============================================================
# Preview Installment-Level Features
# ============================================================

inst_level.select(
    "SK_ID_CURR",
    "SK_ID_PREV",
    "NUM_INSTALMENT_NUMBER",
    "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT",
    "AMT_PAYMENT",
    "Days_Past_Due",
    "Is_Late",
    "Underpaid_Amount"
).show(20, truncate=False)

2026-09-04 21:22:21,349 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:21,863 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:22,925 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:22,941 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:23,135 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:23,136 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:23,139 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:23,301 WARN expressions.RowBasedKeyVal

+----------+----------+---------------------+---------------+------------------+--------------+-----------+-------------+-------+----------------+
|SK_ID_CURR|SK_ID_PREV|NUM_INSTALMENT_NUMBER|DAYS_INSTALMENT|DAYS_ENTRY_PAYMENT|AMT_INSTALMENT|AMT_PAYMENT|Days_Past_Due|Is_Late|Underpaid_Amount|
+----------+----------+---------------------+---------------+------------------+--------------+-----------+-------------+-------+----------------+
|191655    |1818333   |6                    |-345.0         |-356.0            |16479.27      |16479.27   |0.0          |0      |0.0             |
|152976    |2075235   |43                   |-1605.0        |-1619.0           |6750.0        |6750.0     |0.0          |0      |0.0             |
|131846    |1509469   |5                    |-311.0         |-322.0            |1359.99       |1359.99    |0.0          |0      |0.0             |
|120994    |2291733   |3                    |-301.0         |-316.0            |6841.53       |6841.53    |0.0        

In [62]:
# Aggregate installments to customer level
# ============================================================
# Installments Payments - Customer-Level Aggregation
#
# Goal:
# Convert many installments per customer into
# exactly ONE row per SK_ID_CURR.
# ============================================================

inst_agg = (
    inst_level

    .groupBy("SK_ID_CURR")

    .agg(

        # Total number of historical installments
        F.count("*")
         .alias("Total_Installments"),

        # Total amount that should have been paid
        F.round(
            F.sum("AMT_INSTALMENT"),
            2
        ).alias("Total_Expected_Payment"),

        # Total amount actually paid
        F.round(
            F.sum("AMT_PAYMENT"),
            2
        ).alias("Total_Actual_Payment"),

        # Total accumulated days late
        F.round(
            F.sum("Days_Past_Due"),
            2
        ).alias("Total_Days_Past_Due"),

        # Number of installments paid late
        F.sum("Is_Late")
         .alias("Num_Late_Payments"),

        # Average delay length across all installments
        F.round(
            F.avg("Days_Past_Due"),
            2
        ).alias("Avg_Days_Past_Due"),

        # Worst historical payment delay
        F.round(
            F.max("Days_Past_Due"),
            2
        ).alias("Max_Days_Past_Due"),

        # Total amount that remained underpaid
        F.round(
            F.sum("Underpaid_Amount"),
            2
        ).alias("Total_Underpaid")
    )
)

In [63]:
# ============================================================
# Installments Payments - Customer-Level Ratios
# ============================================================

inst_agg = (
    inst_agg

    # Actual amount paid compared with expected amount.
    # 1.0 means total actual payments equal total expected payments.
    .withColumn(
        "Payment_Ratio",
        F.when(
            F.col("Total_Expected_Payment") > 0,
            F.round(
                F.col("Total_Actual_Payment") /
                F.col("Total_Expected_Payment"),
                3
            )
        )
    )

    # Fraction of installments that were paid late.
    # Example:
    # 4 late installments / 20 total = 0.20
    .withColumn(
        "Late_Payment_Ratio",
        F.when(
            F.col("Total_Installments") > 0,
            F.round(
                F.col("Num_Late_Payments") /
                F.col("Total_Installments"),
                3
            )
        )
    )
)

In [64]:
# ============================================================
# Preview Customer-Level Installment Features
# ============================================================

inst_agg.select(
    "SK_ID_CURR",
    "Total_Installments",
    "Total_Expected_Payment",
    "Total_Actual_Payment",
    "Total_Days_Past_Due",
    "Num_Late_Payments",
    "Late_Payment_Ratio",
    "Avg_Days_Past_Due",
    "Max_Days_Past_Due",
    "Total_Underpaid",
    "Payment_Ratio"
).show(10, truncate=False)

2026-09-04 21:22:56,571 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:56,771 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:56,969 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:56,982 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:57,177 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:57,182 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:57,361 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:22:57,374 WARN expressions.RowBasedKeyVal

+----------+------------------+----------------------+--------------------+-------------------+-----------------+------------------+-----------------+-----------------+---------------+-------------+
|SK_ID_CURR|Total_Installments|Total_Expected_Payment|Total_Actual_Payment|Total_Days_Past_Due|Num_Late_Payments|Late_Payment_Ratio|Avg_Days_Past_Due|Max_Days_Past_Due|Total_Underpaid|Payment_Ratio|
+----------+------------------+----------------------+--------------------+-------------------+-----------------+------------------+-----------------+-----------------+---------------+-------------+
|105665    |200               |2055043.84            |2235043.83          |109.0              |8                |0.04              |0.55             |24.0             |0.0            |1.088        |
|109050    |13                |55116.82              |55116.82            |6.0                |1                |0.077             |0.46             |6.0              |0.0            |1.0          |
|1096

In [65]:
# ============================================================
# Validate Customer-Level Grain
#
# Expected:
# No SK_ID_CURR should appear more than once.
# ============================================================

duplicate_customers = (
    inst_agg
    .groupBy("SK_ID_CURR")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate SK_ID_CURR values:", duplicate_customers)

Duplicate SK_ID_CURR values: 0


## Create useful derived features From `bureau`

In [80]:
df_bureau.groupBy("CREDIT_ACTIVE").agg(
         F.count("*")
         .alias("Total_Num"),
).show()

+-------------+---------+
|CREDIT_ACTIVE|Total_Num|
+-------------+---------+
|     Bad debt|       21|
|         Sold|     6527|
|       Active|   630607|
|       Closed|  1079273|
+-------------+---------+



In [49]:
# ============================================================
# Bureau - Customer-Level Aggregation
#
# Goal:
# Convert many bureau credit records per customer into
# exactly ONE row per SK_ID_CURR.
#
# Grain after this step:
# One row = one customer/current application
# ============================================================

bureau_agg = (
    df_bureau

    .groupBy("SK_ID_CURR")

    .agg(

        # Number of distinct external credit accounts
        F.countDistinct("SK_ID_BUREAU")
         .alias("Bureau_Credit_Count"),

        # Number of credits that are currently active
        F.sum(
            F.when(
                F.col("CREDIT_ACTIVE") == "Active",
                1
            ).otherwise(0)
        ).alias("Active_Credit_Count"),

        # Total amount of external credit
        F.round(
            F.sum("AMT_CREDIT_SUM"),
            2
        ).alias("Bureau_Total_Credit"),

        # Total remaining debt with external lenders
        F.round(
            F.sum("AMT_CREDIT_SUM_DEBT"),
            2
        ).alias("Bureau_Total_Debt"),

        # Total amount currently overdue
        F.round(
            F.sum("AMT_CREDIT_SUM_OVERDUE"),
            2
        ).alias("Bureau_Total_Overdue"),

        # Worst number of overdue days among bureau credits
        F.max("CREDIT_DAY_OVERDUE")
         .alias("Bureau_Max_Days_Overdue")
    )
)

In [50]:
# ============================================================
# Bureau - Derived Ratios
# ============================================================

bureau_agg = (
    bureau_agg

    # Percentage of the customer's external credits still active
    .withColumn(
        "Active_Credit_Ratio",
        F.when(
            F.col("Bureau_Credit_Count") > 0,
            F.round(
                F.col("Active_Credit_Count") /
                F.col("Bureau_Credit_Count"),
                3
            )
        )
    )

    # Remaining external debt compared with total external credit
    .withColumn(
        "Debt_to_Credit_Ratio",
        F.when(
            F.col("Bureau_Total_Credit") > 0,
            F.round(
                F.col("Bureau_Total_Debt") /
                F.col("Bureau_Total_Credit"),
                3
            )
        )
    )
)

In [51]:
# ============================================================
# Preview Bureau Features
# ============================================================

bureau_agg.select(
    "SK_ID_CURR",
    "Bureau_Credit_Count",
    "Active_Credit_Count",
    "Active_Credit_Ratio",
    "Bureau_Total_Credit",
    "Bureau_Total_Debt",
    "Debt_to_Credit_Ratio",
    "Bureau_Total_Overdue",
    "Bureau_Max_Days_Overdue"
).show(10, truncate=False)

+----------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+--------------------+-----------------------+
|SK_ID_CURR|Bureau_Credit_Count|Active_Credit_Count|Active_Credit_Ratio|Bureau_Total_Credit|Bureau_Total_Debt|Debt_to_Credit_Ratio|Bureau_Total_Overdue|Bureau_Max_Days_Overdue|
+----------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+--------------------+-----------------------+
|244128    |6                  |4                  |0.667              |4770842.59         |912238.97        |0.191               |0.0                 |0                      |
|160009    |4                  |3                  |0.75               |837346.5           |156960.0         |0.187               |0.0                 |0                      |
|175513    |5                  |3                  |0.6                |609439.5           |0.0              |0.0  

In [52]:
# ============================================================
# Validate Customer-Level Grain
#
# Expected result:
# 0 duplicated SK_ID_CURR values
# ============================================================

duplicate_customers = (
    bureau_agg
    .groupBy("SK_ID_CURR")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate SK_ID_CURR values:", duplicate_customers)

Duplicate SK_ID_CURR values: 0


| Transformed DataFrame | Grain after processing                           | Simple meaning                                                                                                         |
| --------------------- | ------------------------------------------------ | ---------------------------------------------------------------------------------------------------------------------- |
| `df_app_features`     | **1 row per current application (`SK_ID_CURR`)** | Keeps the main current application grain, with added derived features such as income/credit ratios.                    |
| `prev_agg`            | **1 row per `SK_ID_CURR`**                       | All previous Home Credit applications for each customer are summarized into one row.                                   |
| `inst_level`          | **1 row per installment**                        | Multiple payment transactions for the same installment are combined into one installment record.                       |
| `inst_agg`            | **1 row per `SK_ID_CURR`**                       | All historical installments for each customer are summarized into repayment-behavior features.                         |
| `bureau_agg`          | **1 row per `SK_ID_CURR`**                       | All external credit accounts for each customer are summarized into one row.                                            |



## Saving DataFram

In [54]:
df_app_features.printSchema()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- OCCUPATION_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- Employed_to_Age: double (nullable = true)
 |-- Credit_to_Income: double (nullable = true)
 |-- Annuity_to_Income: double (nullable = true)
 |-- LTV: double (nullable = true)
 |-- Credit_to_Annuity: double (nullable = true)



In [57]:
prev_agg.printSchema()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- Prev_App_Count: long (nullable = false)
 |-- Approved_App_Count: long (nullable = false)
 |-- Refused_App_Count: long (nullable = false)
 |-- Prev_Total_Credit: double (nullable = true)
 |-- Prev_Avg_Credit: double (nullable = true)
 |-- Prev_Avg_Annuity: double (nullable = true)
 |-- Prev_Avg_Payment_Term: double (nullable = true)
 |-- Days_Since_Last_Prev_App: integer (nullable = true)
 |-- Approved_App_Ratio: double (nullable = true)
 |-- Refused_App_Ratio: double (nullable = true)



In [66]:
inst_agg.printSchema()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- Total_Installments: long (nullable = false)
 |-- Total_Expected_Payment: double (nullable = true)
 |-- Total_Actual_Payment: double (nullable = true)
 |-- Total_Days_Past_Due: double (nullable = true)
 |-- Num_Late_Payments: long (nullable = true)
 |-- Avg_Days_Past_Due: double (nullable = true)
 |-- Max_Days_Past_Due: double (nullable = true)
 |-- Total_Underpaid: double (nullable = true)
 |-- Payment_Ratio: double (nullable = true)
 |-- Late_Payment_Ratio: double (nullable = true)



In [68]:
bureau_agg.printSchema()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- Bureau_Credit_Count: long (nullable = false)
 |-- Active_Credit_Count: long (nullable = true)
 |-- Bureau_Total_Credit: double (nullable = true)
 |-- Bureau_Total_Debt: double (nullable = true)
 |-- Bureau_Total_Overdue: double (nullable = true)
 |-- Bureau_Max_Days_Overdue: integer (nullable = true)
 |-- Active_Credit_Ratio: double (nullable = true)
 |-- Debt_to_Credit_Ratio: double (nullable = true)



In [69]:
# ============================================================
# Warehouse Ready Columns
# ============================================================

app_curated = df_app_features.select(
    "SK_ID_CURR",
    "TARGET",
    "NAME_CONTRACT_TYPE",

    # Customer dimension attributes
    "OCCUPATION_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "NAME_INCOME_TYPE",
    "FLAG_OWN_REALTY",

    # Loan measures
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",

    # Derived features
    "Employed_to_Age",
    "Credit_to_Income",
    "Annuity_to_Income",
    "LTV",
    "Credit_to_Annuity"
)

In [71]:
base_path = "/user/student/home_credit/curated"

app_curated.coalesce(1).write \
    .mode("overwrite") \
    .parquet(f"{base_path}/application")






In [72]:
prev_agg.coalesce(1).write \
    .mode("overwrite") \
    .parquet(f"{base_path}/previous")

2026-09-04 21:48:35,496 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:48:35,527 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:48:37,066 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


In [74]:
inst_agg.coalesce(1).write \
    .mode("overwrite") \
    .parquet(f"{base_path}/installments")


2026-09-04 21:50:33,490 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:50:33,980 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:50:33,981 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:50:33,986 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:50:34,256 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:50:34,259 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:50:34,545 WARN expressions.RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
2026-09-04 21:50:34,556 WARN expressions.RowBasedKeyVal

In [73]:
bureau_agg.coalesce(1).write \
    .mode("overwrite") \
    .parquet(f"{base_path}/bureau")